# Machine Learning - Assignment 2

**Name:** Praveen Garg<br>
**BITS ID:** 2026AJ05478

## Predicting Online Shoppers' Purchasing Intention

This notebook uses the Online Shoppers Purchasing Intention dataset from the UCI Machine
Learning Repository to predict whether a browsing session on a shopping site ends in a
purchase (`Revenue = True`). It loads and checks the data, preprocesses it, and then
trains and evaluates five classification models. The trained models are saved in
`model/`, the notebook writes `test_data.csv` for the Streamlit app, and it ends with a
table comparing the five models.

In [1]:
# =============================================================================
# Libraries and Global Configuration
# =============================================================================

import pandas as pd

# Define a single fixed random seed used throughout this notebook.
# Fixing it makes the results reproducible, so the final run on the BITS
# Virtual Lab should give the same numbers as my local runs, as long as the
# library versions are the same on both.
RANDOM_STATE = 42

---
## 1. Problem statement

The goal is to predict, for one visit to an online shop, whether the session ends in a
purchase. Each session is described by 17 features. Ten are numeric: how many
administrative, informational and product-related pages the visitor opened and how long
they spent on each kind, three Google Analytics measures of those pages (bounce rate,
exit rate and page value), and how close the visit was to a special day. The other seven
are categorical and describe the visit: the month, operating system, browser, region,
traffic type, visitor type, and whether it fell on a weekend.

This is a binary classification problem. The positive class is a purchase
(`Revenue = True`) and the negative class is no purchase (`Revenue = False`). A shop
would want this prediction while a visitor is still on the site, so it can act on the
sessions that look unlikely to convert, for example with a discount, a live chat prompt
or product recommendations, without handing discounts to people who were going to buy
anyway.

---
## 2. The dataset

I use the Online Shoppers Purchasing Intention Dataset from the
[UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset).
It is published under the
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) licence, which requires the
source to be credited:

- Creators: C. Sakar and Y. Kastro
- DOI: [10.24432/C5F88Q](https://doi.org/10.24432/C5F88Q)
- Paper: Sakar, C.O., Polat, S.O., Katircioglu, M. and Kastro, Y. (2019). *Real-time
  prediction of online shoppers' purchasing intention using multilayer perceptron and
  LSTM recurrent neural networks*. Neural Computing and Applications 31, 6893-6908.

Each row is one browsing session on an e-commerce site. According to the creators, the
data was put together so that each session belongs to a different user over a one-year
period, so that it does not lean towards any particular campaign, special day, user
profile or period.

In [2]:
# =============================================================================
# Load and Inspect Dataset
# =============================================================================

# Load the online shoppers intention dataset from the data directory
csv_file_path = 'data/online_shoppers_intention.csv'
shoppers_df = pd.read_csv(csv_file_path)

# 1. Dataset dimensions
print("=== Dataset Shape ===")
print(f"Shape: {shoppers_df.shape}\n")

# 2. Column data types
print("=== Column Data Types ===")
print(shoppers_df.dtypes)
print()

# 3. Missing values per column
print("=== Missing Values per Column ===")
missing_per_col = shoppers_df.isna().sum()
print(missing_per_col)
print(f"Total missing values: {missing_per_col.sum()}\n")

# 4. Duplicate rows and positive-revenue duplicates
duplicate_rows_mask = shoppers_df.duplicated()
duplicate_rows_count = duplicate_rows_mask.sum()
true_revenue_duplicates = (
    duplicate_rows_mask & shoppers_df['Revenue']
).sum()
print("=== Duplicate Rows Analysis ===")
print(f"Duplicate rows: {duplicate_rows_count}")
print(f"Duplicates with Revenue True: {true_revenue_duplicates}\n")

# 5. Target variable counts and proportion share
revenue_counts = shoppers_df['Revenue'].value_counts()
revenue_shares = shoppers_df['Revenue'].value_counts(normalize=True)
print("=== Target Distribution (Revenue) ===")
print(
    f"Revenue False: {revenue_counts[False]}, "
    f"Revenue True: {revenue_counts[True]}"
)
print(
    f"True share: {revenue_shares[True]:.3f} "
    f"({revenue_shares[True] * 100:.1f}%)"
)
print(
    f"False share: {revenue_shares[False]:.3f} "
    f"({revenue_shares[False] * 100:.1f}%)"
)

=== Dataset Shape ===
Shape: (12330, 18)

=== Column Data Types ===
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object

=== Missing Values per Column ===
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues           

In [3]:
# =============================================================================
# Features and Target Separation
# =============================================================================

# Separate the predictor features (X) from the target outcome variable (y).
# The duplicates stay in for now; I deal with them in the preprocessing step.
X = shoppers_df.drop(columns=['Revenue'])
y = shoppers_df['Revenue']

# Print shapes of the feature matrix and target vector
print("=== Feature Matrix and Target Vector Shapes ===")
print(f"Feature matrix shape (X): {X.shape}")
print(f"Target vector shape (y): {y.shape}\n")

# Group the features by type: 10 numeric, 7 categorical (4 of them coded)
numeric_columns = [
    'Administrative', 'Administrative_Duration', 'Informational',
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]
categorical_columns = [
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend'
]
coded_categorical_columns = [
    'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]

# Make sure the two lists together cover every feature exactly once,
# so no column gets missed or used twice later on
assert sorted(numeric_columns + categorical_columns) == sorted(X.columns)

print("=== Feature Type Breakdown ===")
print(f"Total features: {X.shape[1]}")
print(f"Numeric features: {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")
print(
    "Categorical features stored as number codes: "
    f"{len(coded_categorical_columns)}"
)

# Number of distinct values in each categorical column
print("\n=== Distinct Values per Categorical Column ===")
print(X[categorical_columns].nunique())

=== Feature Matrix and Target Vector Shapes ===
Feature matrix shape (X): (12330, 17)
Target vector shape (y): (12330,)

=== Feature Type Breakdown ===
Total features: 17
Numeric features: 10
Categorical features: 7
Categorical features stored as number codes: 4

=== Distinct Values per Categorical Column ===
Month               10
OperatingSystems     8
Browser             13
Region               9
TrafficType         20
VisitorType          3
Weekend              2
dtype: int64


**Dataset description**

The dataset has 12,330 rows, one per session, and 18 columns. Taking out the target,
`Revenue`, leaves 17 input features, which is well above the minimum of 12 features and
500 instances that the assignment asks for.

Ten of the features are numeric. Six describe the browsing itself: the number of
administrative, informational and product-related pages visited, and the time spent on
each of the three kinds. The other four are the Google Analytics page measures
(`BounceRates`, `ExitRates`, `PageValues`) and `SpecialDay`, which says how close the
visit was to a special day such as Valentine's Day.

The remaining seven are categorical. `Month` (10 distinct months) and `VisitorType`
(`Returning_Visitor`, `New_Visitor` or `Other`) are stored as text, and `Weekend` as
True/False. `OperatingSystems`, `Browser`, `Region` and `TrafficType` are stored as
integers, with 8, 13, 9 and 20 distinct codes, but the numbers are only labels for a
particular operating system, browser, region or traffic source. Browser 4 is not "more"
than browser 2, so these four have to be treated as categories and not as quantities.

The target `Revenue` is True/False, and the positive class is True, a purchase. The
classes are imbalanced: 10,422 sessions (84.5%) ended without a purchase and only 1,908
(15.5%) ended with one.

There are no missing values in any of the 18 columns. There are 125 duplicate rows,
meaning exact copies of an earlier row, and all 125 have `Revenue = False`.

The imbalance changes how I judge the models. A model that predicts "no purchase" for
every session would already get 84.5% accuracy while finding none of the buyers, so its
recall, F1 and MCC would all be 0. Accuracy on its own is misleading here. I still
report it, since the assignment asks for all six metrics, but I compare the models
mainly on recall, F1 and MCC, which only improve when buyers are actually found, and on
AUC, which shows how well a model ranks buyers above non-buyers.